<a href="https://colab.research.google.com/github/PatrickHuynh837/MMAlytics/blob/main/model.ip_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

In [1]:
import os
import sys

import numpy as np
import pandas as pd
import psycopg2
from sqlalchemy import create_engine

from preprocessing import *

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from xgboost import XGBClassifier

DB_URL = (
"postgresql+psycopg://neondb_owner:npg_Bo2SUY6ngypR@"
    "ep-orange-frost-afcl94sd-pooler.c-2.us-west-2.aws.neon.tech/"
    "neondb?sslmode=require"

)

engine = create_engine(
        DB_URL,
        pool_pre_ping=True,
        pool_recycle=3600
        )

In [2]:
#Loading Data
df = pd.read_sql(
    """
    SELECT *
    FROM ml.fight_dataset
    """,
    engine
)

df.head()

,fight_url,event_url,event_name,event_date,location_city,location_state,location_country,referee,weight_class,gender,...,fighter_2_takedown_att,fighter_2_takedown_succ,fighter_2_submission_att,fighter_2_reversals,fighter_2_ctrl_time,fighter_1_rank,fighter_2_rank,fighter_1_odds,fighter_2_odds,created_at
0,http://ufcstats.com/fight-details/c13dc0cccef2...,http://ufcstats.com/event-details/31e1ea6fe6b6...,UFC Fight Night: Fiziev vs. Torres,2026-06-27,Baku,Azerbaijan,None,Marc Goddard,Lightweight,M,...,0.0,0.0,0.0,0.0,0:06,11.0,15.0,NaN,NaN,2026-06-29 05:32:18.095360
1,http://ufcstats.com/fight-details/81cde317c156...,http://ufcstats.com/event-details/31e1ea6fe6b6...,UFC Fight Night: Fiziev vs. Torres,2026-06-27,Baku,Azerbaijan,None,Herb Dean,Flyweight,M,...,0.0,0.0,1.0,0.0,0:00,8.0,14.0,NaN,NaN,2026-06-29 05:32:18.095360
2,http://ufcstats.com/fight-details/809814f03ff3...,http://ufcstats.com/event-details/31e1ea6fe6b6...,UFC Fight Night: Fiziev vs. Torres,2026-06-27,Baku,Azerbaijan,None,Rich Mitchell,Lightweight,M,...,0.0,0.0,0.0,0.0,0:03,NaN,NaN,NaN,NaN,2026-06-29 05:32:18.095360
3,http://ufcstats.com/fight-details/012c307c9d44...,http://ufcstats.com/event-details/31e1ea6fe6b6...,UFC Fight Night: Fiziev vs. Torres,2026-06-27,Baku,Azerbaijan,None,Herb Dean,Middleweight,M,...,2.0,0.0,0.0,0.0,4:04,NaN,14.0,NaN,NaN,2026-06-29 05:32:18.095360
4,http://ufcstats.com/fight-details/6a35ff33b859...,http://ufcstats.com/event-details/31e1ea6fe6b6...,UFC Fight Night: Fiziev vs. Torres,2026-06-27,Baku,Azerbaijan,None,Rich Mitchell,Welterweight,M,...,0.0,0.0,0.0,0.0,0:03,NaN,NaN,NaN,NaN,2026-06-29 05:32:18.095360


# Scope

In [ ]:

#Find missing data
missing = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
      .to_frame('missing_count')
)

missing

In [ ]:
list(df.columns)

# Data Cleaning




## Exploration

### Reach

In [ ]:
missing_fighters = df.loc[
    df["fighter_1_reach_cm"].isna(),
    "fighter_1"
].unique()

fighters_in_both = [
    fighter for fighter in missing_fighters
    if ((df["fighter_1"] == fighter) | (df["fighter_2"] == fighter)).sum() > 1
]

fighters_in_both

In [ ]:
missing_urls = pd.concat([
    df.loc[df["fighter_1_reach_cm"].isna(), "fighter_1_url"],
    df.loc[df["fighter_2_reach_cm"].isna(), "fighter_2_url"]
]).dropna().unique()

print(len(missing_urls))

### Weight

In [ ]:
df[df["fighter_1_weight_lbs"].isna()][
    ["fighter_1","fighter_1_url","weight_class"]
]

In [ ]:
df[df["fighter_2_weight_lbs"].isna()][
    ["fighter_2","fighter_2_url","weight_class"]
]

## Preprocessing

In [3]:
df = preprocess_ranks(df)
df = preprocess_weight(df)

df["fighter_1_win"] = (df["winner"] == df["fighter_1"]).astype(int)

df = df.reset_index(drop=True)
df["fight_id"] = df.index

history = build_fighter_history(df)

df = add_fighter_cumulative_features(df, history)
df = add_striking_rolling_features(df, history)
df = add_grappling_rolling_features(df, history)

In [4]:
df["fighter_1_win"] = (df["winner"] == df["fighter_1"]).astype(int)

# Feature Engineering

## Creation

In [5]:
#Differentials



#Physical
df['height_diff'] = df['fighter_1_height_cm'] - df['fighter_2_height_cm']
df['reach_diff'] = df['fighter_1_reach_cm'] - df['fighter_2_reach_cm']
df['weight_diff'] = df['fighter_1_weight_lbs'] - df['fighter_2_weight_lbs']
df["age_diff"] = (
    (pd.to_datetime(df["event_date"]) - pd.to_datetime(df["fighter_1_dob"])).dt.days -
    (pd.to_datetime(df["event_date"]) - pd.to_datetime(df["fighter_2_dob"])).dt.days
) / 365.25


#Record
df['win_diff'] = (
    df['fighter_1_prior_wins']
    - df['fighter_2_prior_wins']
)

df['loss_diff'] = (
    df['fighter_1_prior_losses']
    - df['fighter_2_prior_losses']
)

df['draw_diff'] = (
    df['fighter_1_prior_draws']
    - df['fighter_2_prior_draws']
)



# Striking (rolling averages already merged)
df["slpm_roll_3_diff"] = (
    df["fighter_1_slpm_roll_3"]
    - df["fighter_2_slpm_roll_3"]
)

df["sapm_roll_3_diff"] = (
    df["fighter_1_sapm_roll_3"]
    - df["fighter_2_sapm_roll_3"]
)

df["str_acc_roll_3_diff"] = (
    df["fighter_1_str_acc_roll_3"]
    - df["fighter_2_str_acc_roll_3"]
)

df["str_def_roll_3_diff"] = (
    df["fighter_1_str_def_roll_3"]
    - df["fighter_2_str_def_roll_3"]
)

# #Grappling
df["td_avg_roll_3_diff"] = (
    df["fighter_1_td_avg_roll_3"]
    - df["fighter_2_td_avg_roll_3"]
)

df["td_acc_roll_3_diff"] = (
    df["fighter_1_td_acc_roll_3"]
    - df["fighter_2_td_acc_roll_3"]
)

df["td_def_roll_3_diff"] = (
    df["fighter_1_td_def_roll_3"]
    - df["fighter_2_td_def_roll_3"]
)

df["sub_avg_roll_3_diff"] = (
    df["fighter_1_sub_avg_roll_3"]
    - df["fighter_2_sub_avg_roll_3"]
)

df["ctrl_time_roll_3_diff"] = (
    df["fighter_1_ctrl_time_roll_3"]
    - df["fighter_2_ctrl_time_roll_3"]
)

df["td_success_rate_roll_3_diff"] = (
    df["fighter_1_td_success_rate_roll_3"]
    - df["fighter_2_td_success_rate_roll_3"]
)

#Perception
df["rank_diff"] = df["fighter_2_rank"] - df["fighter_1_rank"]
df['odds_diff'] = df['fighter_1_odds'] - df['fighter_2_odds']

## Selection

In [ ]:
linear_features = [
    # Physical
    "height_diff",
    "reach_diff",
    "weight_diff",
    "age_diff",

    # Record
    "win_diff",
    "loss_diff",
    "draw_diff",

    # Striking
    "slpm_roll_3_diff",
    "sapm_roll_3_diff",
    "str_acc_roll_3_diff",
    "str_def_roll_3_diff",

    # Grappling
    "td_avg_roll_3_diff",
    "td_acc_roll_3_diff",
    "td_def_roll_3_diff",
    "sub_avg_roll_3_diff",
    "ctrl_time_roll_3_diff",
    "td_success_rate_roll_3_diff",

    # Perception
    "rank_diff",
    "odds_diff"
]

tree_features = [
   
]

df = df.sort_values("event_date")

# Linear model
X_linear = df[linear_features]

# Tree model
X_tree = df[linear_features]

y = df["fighter_1_win"]

## Transformation

# Model Construction


### Logistic Regression

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

def run_model(df, features, target, model, use_scaler=True):
    
    split_idx = int(len(df) * 0.8)
    train_df = df.iloc[:split_idx]
    test_df = df.iloc[split_idx:]

    X_train = train_df[features]
    y_train = train_df[target]

    X_test = test_df[features]
    y_test = test_df[target]

    steps = [("imputer", SimpleImputer(strategy="median"))]
    if use_scaler:
        steps.append(("scaler", StandardScaler()))
    steps.append(("model", model))
    pipeline = Pipeline(steps)

    pipeline.fit(X_train, y_train)

    preds = pipeline.predict(X_test)
    probs = pipeline.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, preds)

    return pipeline, acc, preds, probs, y_test

# Backward-compatible wrapper for the existing call sites.
def run_logistic_regression(df, features, target):
    return run_model(
        df,
        features,
        target,
        LogisticRegression(
            class_weight='balanced',
            max_iter=1000,
            random_state=42
        ),
        use_scaler=True
    )


In [9]:
# 1. Unpack all 5 returned values (including y_test)
lr_model, lr_acc, lr_preds, lr_probs, y_test = run_logistic_regression(
    df,
    linear_features,
    "fighter_1_win"
) 

# 2. Use the correctly prefixed variables (lr_preds, lr_probs)
print(f"Accuracy:  {accuracy_score(y_test, lr_preds):.4f}")
print(f"Precision: {precision_score(y_test, lr_preds):.4f}")
print(f"Recall:    {recall_score(y_test, lr_preds):.4f}")
print(f"F1 Score:  {f1_score(y_test, lr_preds):.4f}")
print(classification_report(y_test, lr_preds))
print(f"ROC AUC:   {roc_auc_score(y_test, lr_probs):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, lr_preds))

Accuracy:  0.6592
Precision: 0.6946
Recall:    0.6795
F1 Score:  0.6869
              precision    recall  f1-score   support

           0       0.62      0.63      0.63       788
           1       0.69      0.68      0.69       964

    accuracy                           0.66      1752
   macro avg       0.66      0.66      0.66      1752
weighted avg       0.66      0.66      0.66      1752

ROC AUC:   0.7177

Confusion Matrix:
[[500 288]
 [309 655]]


### Boosted Trees

In [14]:
from sklearn.ensemble import RandomForestClassifier

rf_model, rf_acc, rf_preds, rf_probs, y_test = run_model(
    df,
    linear_features,
    "fighter_1_win",
    RandomForestClassifier(random_state=42, class_weight='balanced'),
    use_scaler=False
)


In [15]:
print(f"Accuracy:  {accuracy_score(y_test, rf_preds):.4f}")
print(f"Precision: {precision_score(y_test, rf_preds):.4f}")
print(f"Recall:    {recall_score(y_test, rf_preds):.4f}")
print(f"F1 Score:  {f1_score(y_test, rf_preds):.4f}")
print(classification_report(y_test, rf_preds))
print(f"ROC AUC:   {roc_auc_score(y_test, rf_probs):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, rf_preds))

Accuracy:  0.6130
Precision: 0.6059
Recall:    0.8485
F1 Score:  0.7070
              precision    recall  f1-score   support

           0       0.64      0.32      0.43       788
           1       0.61      0.85      0.71       964

    accuracy                           0.61      1752
   macro avg       0.62      0.59      0.57      1752
weighted avg       0.62      0.61      0.58      1752

ROC AUC:   0.6765

Confusion Matrix:
[[256 532]
 [146 818]]


In [ ]:
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score

def run_xgboost(df, features, target, scale_pos_weight=None):

    split_idx = int(len(df) * 0.8)
    train_df = df.iloc[:split_idx]
    test_df = df.iloc[split_idx:]

    X_train = train_df[features]
    y_train = train_df[target]

    X_test = test_df[features]
    y_test = test_df[target]

    model = XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        min_child_weight=5,
        gamma=0.1,
        random_state=42,
        eval_metric="logloss",
        scale_pos_weight=scale_pos_weight
    )

    pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    preds = pipeline.predict(X_test)
    probs = pipeline.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, preds)

    # Added y_test to the return statement so you can use it for your metrics below
    return pipeline, acc, preds, probs, y_test

In [12]:
# Unpack all 5 variables, using your tree_features list
xgb_model, xgb_acc, xgb_preds, xgb_probs, y_test = run_xgboost(
    df,
    linear_features, 
    "fighter_1_win",
    scale_pos_weight=1.5
) 

# Print out your evaluation metrics
print(f"Accuracy:  {accuracy_score(y_test, xgb_preds):.4f}")
print(f"Precision: {precision_score(y_test, xgb_preds):.4f}")
print(f"Recall:    {recall_score(y_test, xgb_preds):.4f}")
print(f"F1 Score:  {f1_score(y_test, xgb_preds):.4f}")
print(classification_report(y_test, xgb_preds))
print(f"ROC AUC:   {roc_auc_score(y_test, xgb_probs):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, xgb_preds))

Accuracy:  0.6239
Precision: 0.6081
Recall:    0.8900
F1 Score:  0.7225
              precision    recall  f1-score   support

           0       0.69      0.30      0.42       788
           1       0.61      0.89      0.72       964

    accuracy                           0.62      1752
   macro avg       0.65      0.59      0.57      1752
weighted avg       0.64      0.62      0.58      1752

ROC AUC:   0.6704

Confusion Matrix:
[[235 553]
 [106 858]]


d

In [18]:
list(df.columns)

['fight_url',
 'event_url',
 'event_name',
 'event_date',
 'location_city',
 'location_state',
 'location_country',
 'referee',
 'weight_class',
 'gender',
 'title_fight',
 'num_rounds',
 'fighter_1',
 'fighter_2',
 'fighter_1_url',
 'fighter_2_url',
 'winner',
 'result',
 'result_details',
 'finish_round',
 'finish_time',
 'fighter_1_height_cm',
 'fighter_1_weight_lbs',
 'fighter_1_reach_cm',
 'fighter_1_stance',
 'fighter_1_dob',
 'fighter_2_height_cm',
 'fighter_2_weight_lbs',
 'fighter_2_reach_cm',
 'fighter_2_stance',
 'fighter_2_dob',
 'fighter_1_wins',
 'fighter_1_losses',
 'fighter_1_draws',
 'fighter_1_slpm',
 'fighter_1_str_acc',
 'fighter_1_sapm',
 'fighter_1_str_def',
 'fighter_1_td_avg',
 'fighter_1_td_acc',
 'fighter_1_td_def',
 'fighter_1_sub_avg',
 'fighter_2_wins',
 'fighter_2_losses',
 'fighter_2_draws',
 'fighter_2_slpm',
 'fighter_2_str_acc',
 'fighter_2_sapm',
 'fighter_2_str_def',
 'fighter_2_td_avg',
 'fighter_2_td_acc',
 'fighter_2_td_def',
 'fighter_2_sub_avg',

In [17]:
y.value_counts()

fighter_1_win
1    5519
0    3239
Name: count, dtype: int64